In [26]:
import pandas as pd
import requests
from sqlalchemy import create_engine

In [27]:
engine = create_engine("mysql+pymysql://root@localhost/ecommercedatabase")
# ── EXTRACT ──────────────────────────────────────────────
products_raw = requests.get("https://dummyjson.com/products?limit=100").json()['products']
users_raw    = requests.get("https://dummyjson.com/users?limit=100").json()['users']
carts_raw    = requests.get("https://dummyjson.com/carts?limit=100").json()['carts']

In [28]:


# ── TRANSFORM ─────────────────────────────────────────────

# Products - keep only flat columns
df_products = pd.DataFrame(products_raw)[[
    'id', 'title', 'price', 'category', 'stock', 
    'rating', 'brand', 'discountPercentage'
]]

# Users - flatten nested address
df_users = pd.DataFrame(users_raw)
df_users['city']  = df_users['address'].apply(lambda x: x['city'])
df_users['state'] = df_users['address'].apply(lambda x: x['state'])
df_users = df_users[['id', 'firstName', 'lastName', 'email', 'city', 'state']]

# Carts - keep only flat columns
df_carts = pd.DataFrame(carts_raw)[[
    'id', 'userId', 'totalProducts', 'totalQuantity', 
    'total', 'discountedTotal'
]]

print("Products shape:", df_products.shape)
print("Users shape:", df_users.shape)
print("Carts shape:", df_carts.shape)

print(df_products.dtypes)
print(df_users.dtypes)
print(df_carts.dtypes)

Products shape: (100, 8)
Users shape: (100, 6)
Carts shape: (100, 6)
id                      int64
title                  object
price                 float64
category               object
stock                   int64
rating                float64
brand                  object
discountPercentage    float64
dtype: object
id            int64
firstName    object
lastName     object
email        object
city         object
state        object
dtype: object
id                   int64
userId               int64
totalProducts        int64
totalQuantity        int64
total              float64
discountedTotal    float64
dtype: object


In [29]:



df_products.to_sql('products', con=engine, if_exists='replace', index=False)
df_users.to_sql('users', con=engine, if_exists='replace', index=False)
df_carts.to_sql('carts', con=engine, if_exists='replace', index=False)

print("All tables loaded successfully")

All tables loaded successfully
